In [5]:
import os
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset, WeightedRandomSampler
from torchvision import transforms
from sklearn.metrics import confusion_matrix, classification_report, cohen_kappa_score, accuracy_score
from PIL import Image
import timm

# --- CONFIG ---
# Force single GPU usage to prevent the internal PyTorch crash
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
BASE_PATH = '/kaggle/input/datasets/mariaherrerot/aptos2019/'
TRAIN_IMG_DIR = os.path.join(BASE_PATH, 'train_images/train_images')
VAL_IMG_DIR = os.path.join(BASE_PATH, 'val_images/val_images')
IMG_SIZE = 256
BATCH_SIZE = 32
LR = 1e-4
EPOCHS = 20

# --- NOVELTY: CBAM (Convolutional Block Attention Module) ---
class CBAM(nn.Module):
    def __init__(self, channels, reduction=16):
        super(CBAM, self).__init__()
        # Channel Attention: Focusing on 'What' (color/feature types)
        self.avg_pool = nn.AdaptiveAvgPool2d(1)
        self.max_pool = nn.AdaptiveMaxPool2d(1)
        self.fc = nn.Sequential(
            nn.Linear(channels, channels // reduction, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(channels // reduction, channels, bias=False)
        )
        # Spatial Attention: Focusing on 'Where' (lesion location)
        self.spatial_conv = nn.Conv2d(2, 1, kernel_size=7, padding=3, bias=False)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        b, c, _, _ = x.size()
        # Channel attention
        avg_out = self.fc(self.avg_pool(x).view(b, c)).view(b, c, 1, 1)
        max_out = self.fc(self.max_pool(x).view(b, c)).view(b, c, 1, 1)
        x = x * self.sigmoid(avg_out + max_out)
        # Spatial attention
        avg_s = torch.mean(x, dim=1, keepdim=True)
        max_s, _ = torch.max(x, dim=1, keepdim=True)
        s_att = self.sigmoid(self.spatial_conv(torch.cat([avg_s, max_s], dim=1)))
        return x * s_att

# --- HYBRID ARCHITECTURE ---
class AttentiveAptosNet(nn.Module):
    def __init__(self):
        super().__init__()
        # EfficientNet-V2-S: Specifically chosen for its ability to handle fine-grained textures
        self.backbone = timm.create_model('tf_efficientnetv2_s', pretrained=True, num_classes=0, global_pool='')
        self.attention = CBAM(1280)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(1280, 512),
            nn.BatchNorm1d(512),
            nn.Hardswish(),
            nn.Dropout(0.5), # Increased dropout for better generalization
            nn.Linear(512, 5)
        )

    def forward(self, x):
        x = self.backbone(x)
        x = self.attention(x)
        x = self.pool(x)
        return self.head(x)

# --- DATASET LOADER ---
class AptosDataset(Dataset):
    def __init__(self, csv_file, img_dir, transform=None):
        self.df = pd.read_csv(os.path.join(BASE_PATH, csv_file))
        self.img_dir = img_dir
        self.transform = transform
    def __len__(self): return len(self.df)
    def __getitem__(self, idx):
        img_path = os.path.join(self.img_dir, f"{self.df.iloc[idx, 0]}.png")
        image = Image.open(img_path).convert('RGB')
        label = self.df.iloc[idx, 1]
        if self.transform: image = self.transform(image)
        return image, torch.tensor(label, dtype=torch.long)

# --- ENGINE ---
def run_experiment():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Executing on: {device}")

    model = AttentiveAptosNet().to(device)

    # Weighted Sampler to fix the 82% plateau by forcing minority class training
    train_df = pd.read_csv(os.path.join(BASE_PATH, 'train_1.csv'))
    class_counts = train_df.diagnosis.value_counts()
    weights = 1. / class_counts
    sample_weights = weights[train_df.diagnosis].values
    sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

    train_tf = transforms.Compose([
        transforms.Resize((IMG_SIZE, IMG_SIZE)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.RandomRotation(10),
        transforms.ColorJitter(0.2, 0.2, 0.1),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
    ])

    train_loader = DataLoader(AptosDataset('train_1.csv', TRAIN_IMG_DIR, train_tf), 
                              batch_size=BATCH_SIZE, sampler=sampler, num_workers=4)
    val_loader = DataLoader(AptosDataset('valid.csv', VAL_IMG_DIR, train_tf), 
                            batch_size=BATCH_SIZE, num_workers=4)

    optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-2)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    
    best_kappa = 0
    for epoch in range(EPOCHS):
        model.train()
        for imgs, labels in train_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            optimizer.zero_grad()
            loss = criterion(model(imgs), labels)
            loss.backward()
            optimizer.step()

        # Evaluation
        model.eval()
        v_preds, v_labels = [], []
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs, labels = imgs.to(device), labels.to(device)
                v_preds.extend(torch.argmax(model(imgs), 1).cpu().numpy())
                v_labels.extend(labels.cpu().numpy())
        
        kappa = cohen_kappa_score(v_labels, v_preds, weights='quadratic')
        acc = accuracy_score(v_labels, v_preds)
        print(f"Epoch {epoch+1} | Accuracy: {acc:.4f} | Kappa: {kappa:.4f}")
        
        if kappa > best_kappa:
            best_kappa = kappa
            torch.save(model.state_dict(), 'best_attentive_model.pth')

    # FINAL PERFORMANCE DOCUMENTATION
    print("\n" + "="*60)
    print("             FINAL CLINICAL PERFORMANCE REPORT")
    print("="*60)
    model.load_state_dict(torch.load('best_attentive_model.pth'))
    model.eval()
    
    f_preds, f_labels = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device)
            f_preds.extend(torch.argmax(model(imgs), 1).cpu().numpy())
            f_labels.extend(labels.cpu().numpy())

    cm = confusion_matrix(f_labels, f_preds)
    classes = ['No DR', 'Mild', 'Moderate', 'Severe', 'Proliferative']
    
    print(f"Final Accuracy: {accuracy_score(f_labels, f_preds):.4%}")
    print(f"Final Kappa:    {cohen_kappa_score(f_labels, f_preds, weights='quadratic'):.4f}")
    print("-" * 60)
    
    for i in range(5):
        tp = cm[i, i]
        fn = sum(cm[i, :]) - tp
        fp = sum(cm[:, i]) - tp
        tn = sum(cm.flatten()) - (tp + fn + fp)
        sens = tp / (tp + fn) if (tp + fn) > 0 else 0
        spec = tn / (tn + fp) if (tn + fp) > 0 else 0
        print(f"{classes[i]:<15} | Sensitivity: {sens:.4f} | Specificity: {spec:.4f}")
    
    print("\nFull Classification Report:\n", classification_report(f_labels, f_preds, target_names=classes))

if __name__ == "__main__":
    run_experiment()

Executing on: cuda
Epoch 1 | Accuracy: 0.7432 | Kappa: 0.8481
Epoch 2 | Accuracy: 0.7787 | Kappa: 0.8375
Epoch 3 | Accuracy: 0.7814 | Kappa: 0.8475
Epoch 4 | Accuracy: 0.7951 | Kappa: 0.8544
Epoch 5 | Accuracy: 0.7923 | Kappa: 0.8726
Epoch 6 | Accuracy: 0.7951 | Kappa: 0.8806
Epoch 7 | Accuracy: 0.8033 | Kappa: 0.8654
Epoch 8 | Accuracy: 0.8033 | Kappa: 0.8714
Epoch 9 | Accuracy: 0.8060 | Kappa: 0.8748
Epoch 10 | Accuracy: 0.8224 | Kappa: 0.8913
Epoch 11 | Accuracy: 0.8306 | Kappa: 0.8865
Epoch 12 | Accuracy: 0.8279 | Kappa: 0.8892
Epoch 13 | Accuracy: 0.8142 | Kappa: 0.8940
Epoch 14 | Accuracy: 0.8279 | Kappa: 0.8960
Epoch 15 | Accuracy: 0.8388 | Kappa: 0.8915
Epoch 16 | Accuracy: 0.8115 | Kappa: 0.8609
Epoch 17 | Accuracy: 0.8388 | Kappa: 0.8902
Epoch 18 | Accuracy: 0.8361 | Kappa: 0.8916
Epoch 19 | Accuracy: 0.8388 | Kappa: 0.8904
Epoch 20 | Accuracy: 0.8279 | Kappa: 0.8792

             FINAL CLINICAL PERFORMANCE REPORT
Final Accuracy: 81.6940%
Final Kappa:    0.8793
--------------